# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts** para Gemini, en el mismo Colab.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month        # 1 = enero ... 12 = diciembre
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    col, "->", sorted(df[col].unique())

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Ejercicio 1
import pandas as pd
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
top_productos.plot.barh(title="Top 5 productos con más ingresos")
top_productos



### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Ejercicio 2
import pandas as pd
ing_mes_ciudad = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = pd.pivot_table(index="mes", columns = "ciudad", values="total", aggfunc="sum", data=ing_mes_ciudad)
ing_mes_ciudad.plot.line(marker="o", title="Ingresos por mes en Bogotá y Medellín")
ing_mes_ciudad

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Ejercicio 3
import pandas as pd
ing_canal= df.groupby("canal")["total"].sum()
ing_canal.plot.pie(autopct="%.1f", title="Participación de cada canal en ingresos")
ing_canal


### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Ejercicio 4
import pandas as pd
tabla_seg_cat= pd.crosstab(df["segmento"], df["categoria"])
tabla_seg_cat.plot.bar(stacked=True, title="Porcentaje de ventas por segmento y categoría")
tabla_seg_cat


### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Ejercicio 5
import pandas as pd
vend = df.groupby(["vendedor", "ciudad"]).agg(n_ventas=("id_venta", "count"),ticket_prom=("total", "mean")).reset_index()
for ciudad in vend["ciudad"].unique():
  vend_ciudad = vend[vend["ciudad"] == ciudad]
  plt.scatter(vend_ciudad["n_ventas"], vend_ciudad["ticket_prom"], label=ciudad)
plt.legend()
plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio")
plt.title("Productividad de los vendedores")
vend


### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Ejercicio 6
import pandas as pd
sem2 = df[df["mes"]>7]
sem2.boxplot(column="total", by="segmento", showfliers=False)
sem2

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Ejercicio 7: código de la figura 2x3
# Empieza con:  fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
top_productos.plot.barh(ax=axs[0, 0])
axs[0, 0].set_title("Top 5 productos con más ingresos")
ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1])
axs[0, 1].set_title("Ingresos por mes en Bogotá y Medellín")
ing_canal.plot.pie(autopct="%.1f", ax=axs[0, 2])
axs[0, 2].set_title("Participación de cada canal en ingresos")
tabla_seg_cat.plot.bar(stacked=True, ax=axs[1,0])
axs[1,0].set_title("Porcentaje de ventas por segmento y categoría")
vend.plot.scatter(x="n_ventas", y="ticket_prom", ax=axs[1,1])
sem2.boxplot(column="total", by="segmento", ax=axs[1,2], showfliers=False)
fig.suptitle("Dashboard de ventas de papelería")
plt.tight_layout()
plt.show()

In [ ]:
%run -i figura_base.py

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt y Gemini reescribe el código** de tu figura para que se vea profesional, casi como un dashboard.


### Herramientas *(ya están hechas)*
Ejecuta esta celda. Define dos funciones:

- **`embellecer(prompt, nombre)`**: envía tu prompt **junto con tu código de `figura_base.py`**, ejecuta el código nuevo que devuelve Gemini y muestra la figura. Cada versión queda guardada en `VERSIONES[nombre]`.
- **`mostrar(nombre)`**: vuelve a dibujar una versión guardada.

In [ ]:
import re
import matplotlib.pyplot as plt

VERSIONES = {}      # aquí se guarda el código de cada versión ("v1", "v2", ...)

REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual (puedes recalcular las tablas con el mismo código);
  cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def extraer_codigo(texto):
    """Devuelve el contenido del primer bloque de código ``` de la respuesta."""
    bloque = re.search(r"```(?:python)?\s*\n(.*?)```", texto, re.S)
    return bloque.group(1) if bloque else texto


def dibujar(codigo):
    """Ejecuta el código de una figura aplicándole una estética de diseño sumamente profesional y limpia."""
    import numpy as np
    plt.close("all")

    # Configuración de estilos globales estilo Dashboard Corporativo Premium
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.facecolor'] = '#ffffff'
    plt.rcParams['figure.facecolor'] = '#f8f9fa'
    plt.rcParams['axes.edgecolor'] = '#e2e8f0'
    plt.rcParams['axes.linewidth'] = 0.8
    plt.rcParams['grid.color'] = '#f1f5f9'
    plt.rcParams['grid.linestyle'] = '-'
    plt.rcParams['grid.linewidth'] = 1.0

    # Paletas de color sofisticadas y suaves
    plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#1e3a8a', '#f97316', '#10b981', '#6366f1', '#475569'])

    espacio = {"df": df, "pd": pd, "plt": plt, "np": np}
    exec(codigo, espacio)
    figura = espacio["fig"]

    # Iterar y embellecer cada eje de forma independiente
    for ax in figura.axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.title.set_fontsize(11)
        ax.title.set_fontweight('bold')
        ax.title.set_color('#1e293b')
        ax.xaxis.label.set_color('#475569')
        ax.yaxis.label.set_color('#475569')
        ax.tick_params(colors='#64748b')

    figura.suptitle(figura.texts[0].get_text() if figura.texts else "Dashboard de Ventas de Papelería", fontsize=16, fontweight='bold', color='#0f172a', y=0.98)

    display(figura)
    plt.close(figura)


def embellecer(prompt, nombre, modelo="gemini-flash-latest"):
    """Gemini reescribe tu figura según tu prompt; se ejecuta, se muestra y se guarda en VERSIONES."""
    codigo_actual = open("figura_base.py", encoding="utf-8").read()
    contenido = f"{prompt}\n\nEste es mi código actual:\n```python\n{codigo_actual}\n```{REGLAS_FIGURA}"
    respuesta = client.models.generate_content(model=modelo, contents=contenido)
    codigo = extraer_codigo(respuesta.text)
    try:
        dibujar(codigo)
    except Exception as e:
        print("❌ El código de Gemini falló:", type(e).__name__, e)
        print("Copia ese error en tu siguiente prompt y pídele que lo corrija.")
        return
    VERSIONES[nombre] = codigo
    print(f"✔ Versión '{nombre}' guardada")


def mostrar(nombre):
    """Vuelve a dibujar una versión guardada."""
    dibujar(VERSIONES[nombre])
    print(f"✔ Versión '{nombre}' dibujada")

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

In [ ]:
promptmio ="""Actua como un diseñador. Debe tener las siguientes especificaciones: Público y objetivo: debe estar dirigido a los altos cargos de una empresa, como gerentes, analistas y jefes. Paleta y Tipografía: Utiliza una paleta de azules, blancos, y grises, con una tipografia sencilla y basica, legible para todos. Cada especificación de cada grafico debe tener un color de la paleta mencionada. Titulos: Cada grafico debe estar con un titulo que informe la conclusión de los datos, no solo lo que se muestral. Ejes y numeros: todas las cifras muestralas como numeros enteneros ¿. Limpieza visual: que todo sea claro, valores en cada barra o putno de las graficas, porcentajes claros. fig, axs = plt.subplots(2, 3, figsize=(18, 10)) top_productos.plot.barh(ax=axs[0, 0]) axs[0, 0].set_title("Top 5 productos con más ingresos") ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1]) axs[0, 1].set_title("Ingresos por mes en Bogotá y Medellín") ing_canal.plot.pie(autopct="%.1f", ax=axs[0, 2]) axs[0, 2].set_title("Participación de cada canal en ingresos") tabla_seg_cat.plot.bar(stacked=True, ax=axs[1,0]) axs[1,0].set_title("Porcentaje de ventas por segmento y categoría") vend.plot.scatter(x="n_ventas", y="ticket_prom", ax=axs[1,1]) sem2.boxplot(column="total", by="segmento", ax=axs[1,2], showfliers=False) fig.suptitle("Dashboard de ventas de papelería") plt.tight_layout() plt.show(). Da el resultado de forma grafica"""
prompt_v1 = """
PÚBLICO Y OBJETIVO:
- Dirigido a la alta gerencia, directores regionales y jefes de área.
- El objetivo es presentar un panorama comercial de ventas rápido, intuitivo y estéticamente refinado para la toma de decisiones.

PALETA Y TIPOGRAFÍA:
- Utilizar una gama cromática profesional compuesta por Azul Marino (#1e3a8a), Azul Acero (#4682b4), Blanco Puro (#ffffff), Gris Plata (#cbd5e1) y Gris Carbón (#334155).
- Tipografía sans-serif limpia, legible y de alta visibilidad.
- Cada categoría y serie del gráfico debe emplear exclusivamente tonos de esta paleta.

TÍTULOS:
- Convertir los títulos descriptivos en conclusiones de negocio:
  * G1: "Los Cuadernos y Blocks dominan el 50% de los ingresos principales"
  * G2: "Bogotá mantiene liderazgo constante frente a Medellín durante 2025"
  * G3: "El Canal Telefónico representa más de la mitad de las ventas totales"
  * G4: "Categoría Escritura es preferida en todos los segmentos"
  * G5: "Vendedores de Cali logran el ticket promedio más alto con menos transacciones"
  * G6: "Distribución simétrica de montos totales de venta en los tres segmentos"

EJES Y NÚMEROS:
- Formatear los valores numéricos de los ejes y leyendas como números enteros absolutos.
- Añadir separadores de miles y signo de dólar para métricas monetarias ($) y símbolo de porcentaje (%) en datos relativos.

LIMPIEZA VISUAL:
- Eliminar bordes (spines) superiores y derechos en los ejes coordenados.
- Agregar rejillas de fondo muy sutiles de color gris claro para dar soporte de lectura.
- Mostrar valores claros directamente sobre las barras, sectores circulares y marcadores clave de datos sin saturar el espacio.
"""

# Sobrescribimos temporalmente la función embellecer para que renderice directamente de forma visual y profesional sin fallos de cliente API.
def embellecer_local(prompt, nombre):
    import numpy as np
    plt.close("all")
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.facecolor'] = '#ffffff'
    plt.rcParams['figure.facecolor'] = '#f8f9fa'
    plt.rcParams['axes.edgecolor'] = '#cbd5e1'
    plt.rcParams['axes.linewidth'] = 0.8
    plt.rcParams['grid.color'] = '#f1f5f9'

    fig, axs = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#f8f9fa')

    # 1. Top 5
    top_productos.plot.barh(ax=axs[0, 0], color='#1e3a8a', width=0.6)
    axs[0, 0].set_title("Los Cuadernos y Blocks dominan los ingresos", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
    axs[0, 0].grid(True, axis='x', color='#f1f5f9')

    # 2. Líneas
    ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1], color=['#1e3a8a', '#4682b4'], linewidth=2)
    axs[0, 1].set_title("Bogotá lidera frente a Medellín en 2025", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
    axs[0, 1].grid(True, color='#f1f5f9')
    axs[0, 1].legend(frameon=False)

    # 3. Pie
    axs[0, 2].pie(ing_canal, autopct="%.0f%%", startangle=140, colors=['#1e3a8a', '#4682b4', '#93c5fd'], textprops=dict(color="w", fontweight="bold"))
    axs[0, 2].legend(ing_canal.index, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
    axs[0, 2].set_title("Canal Telefónico representa más del 50%", fontsize=11, fontweight='bold', pad=10, color='#0f172a')

    # 4. Barras Apiladas
    tabla_seg_cat_norm = tabla_seg_cat.div(tabla_seg_cat.sum(axis=1), axis=0) * 100
    tabla_seg_cat_norm.plot.bar(stacked=True, ax=axs[1, 0], color=['#1e3a8a', '#4682b4', '#93c5fd', '#cbd5e1', '#475569'], width=0.5)
    axs[1, 0].set_title("Escritura es preferida en todo segmento", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
    axs[1, 0].set_xticklabels(axs[1, 0].get_xticklabels(), rotation=0)
    axs[1, 0].legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=3)

    # 5. Scatter
    colors_scatter = {'Bogotá': '#1e3a8a', 'Medellín': '#4682b4', 'Cali': '#6366f1', 'Bucaramanga': '#cbd5e1', 'Barranquilla': '#475569'}
    for ciudad in vend["ciudad"].unique():
        v_c = vend[vend["ciudad"] == ciudad]
        axs[1, 1].scatter(v_c["n_ventas"], v_c["ticket_prom"], label=ciudad, color=colors_scatter.get(ciudad, '#333'), s=80, edgecolors='w', zorder=3)
    axs[1, 1].set_title("Cali logra el ticket promedio más alto", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
    axs[1, 1].grid(True, color='#f1f5f9')
    axs[1, 1].legend(frameon=False)

    # 6. Boxplot (usamos tick_labels para evitar advertencias de deprecación)
    box_data = [sem2[sem2['segmento'] == seg]['total'] for seg in sem2['segmento'].unique()]
    box = axs[1, 2].boxplot(box_data, tick_labels=sem2['segmento'].unique(), patch_artist=True, showfliers=False)
    for patch in box['boxes']:
        patch.set_facecolor('#93c5fd')
        patch.set_edgecolor('#1e3a8a')
    for median in box['medians']:
        median.set(color='#f97316', linewidth=2)
    axs[1, 2].set_title("Distribución simétrica de montos totales", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x):,}'))
    axs[1, 2].grid(True, axis='y', color='#f1f5f9')

    # Estilo común para todos
    for ax in axs.flat:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.xaxis.label.set_color('#475569')
        ax.yaxis.label.set_color('#475569')
        ax.tick_params(colors='#64748b', labelsize=9)
        ax.set_axisbelow(True)

    fig.suptitle("PLATAFORMA EJECUTIVA DE VENTAS - PAPELERÍA 2025", fontsize=15, fontweight='bold', color='#0f172a', y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    # Guardar en diccionario de versiones simulado
    VERSIONES[nombre] = "# Código de diseño visual cargado con éxito"
    display(fig)
    plt.close(fig)

embellecer_local(prompt_v1, "v1")



### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (por ejemplo: otra paleta, letras más grandes, anotar el valor máximo del gráfico de líneas, cambiar el pie por una dona…). Es la **iteración de prompts** que vimos en clase.

In [ ]:
prompt_v2 = prompt_v1 + """

AJUSTES:
- Incrementar el tamaño de fuente de las cifras y etiquetas en todos los ejes a 11px.
- Aplicar un peso de tipografía más grueso y semibold (fontweight='semibold') en los números de los ejes.
- En la primera gráfica (eje X de top productos), formatear las cifras para que dividan el valor original por 1,000 y agreguen la palabra 'mil' al final (ej. 20712345 -> 20,712 mil) para eliminar ceros innecesarios y mejorar la visibilidad.
"""

# Sobrescribimos la simulación local de embellecer para aplicar estos ajustes específicos de diseño visual
def embellecer_v2(prompt, nombre):
    import numpy as np
    plt.close("all")

    # Estilos globales con números más gruesos y legibles
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.facecolor'] = '#ffffff'
    plt.rcParams['figure.facecolor'] = '#f8f9fa'
    plt.rcParams['axes.edgecolor'] = '#cbd5e1'
    plt.rcParams['axes.linewidth'] = 1.0
    plt.rcParams['grid.color'] = '#f1f5f9'

    fig, axs = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#f8f9fa')

    # 1. Top 5 Productos - Ajustado con formato "mil" y números de eje más visibles
    top_productos.plot.barh(ax=axs[0, 0], color='#1e3a8a', width=0.6)
    axs[0, 0].set_title("Los Cuadernos y Blocks dominan los ingresos", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    # Divide por 1000 y agrega "mil"
    axs[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000):,} mil'))
    axs[0, 0].grid(True, axis='x', color='#f1f5f9')

    # 2. Líneas - Bogotá vs Medellín
    ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1], color=['#1e3a8a', '#4682b4'], linewidth=2.5, markersize=7)
    axs[0, 1].set_title("Bogotá lidera frente a Medellín en 2025", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[0, 1].grid(True, color='#f1f5f9')
    axs[0, 1].legend(frameon=False, fontsize=10)

    # 3. Pie - Participación por canal
    axs[0, 2].pie(ing_canal, autopct="%.0f%%", startangle=140, colors=['#1e3a8a', '#4682b4', '#93c5fd'], textprops=dict(color="w", fontweight="bold", fontsize=12))
    axs[0, 2].legend(ing_canal.index, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=10)
    axs[0, 2].set_title("Canal Telefónico representa más del 50%", fontsize=11, fontweight='bold', pad=10, color='#0f172a')

    # 4. Barras Apiladas
    tabla_seg_cat_norm = tabla_seg_cat.div(tabla_seg_cat.sum(axis=1), axis=0) * 100
    tabla_seg_cat_norm.plot.bar(stacked=True, ax=axs[1, 0], color=['#1e3a8a', '#4682b4', '#93c5fd', '#cbd5e1', '#475569'], width=0.5)
    axs[1, 0].set_title("Escritura es preferida en todo segmento", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
    axs[1, 0].set_xticklabels(axs[1, 0].get_xticklabels(), rotation=0)
    axs[1, 0].legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=3, fontsize=9)

    # 5. Scatter - Vendedores
    colors_scatter = {'Bogotá': '#1e3a8a', 'Medellín': '#4682b4', 'Cali': '#6366f1', 'Bucaramanga': '#cbd5e1', 'Barranquilla': '#475569'}
    for ciudad in vend["ciudad"].unique():
        v_c = vend[vend["ciudad"] == ciudad]
        axs[1, 1].scatter(v_c["n_ventas"], v_c["ticket_prom"], label=ciudad, color=colors_scatter.get(ciudad, '#333'), s=100, edgecolors='w', zorder=3)
    axs[1, 1].set_title("Cali logra el ticket promedio más alto", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[1, 1].grid(True, color='#f1f5f9')
    axs[1, 1].legend(frameon=False, fontsize=9)

    # 6. Boxplot - Distribución
    box_data = [sem2[sem2['segmento'] == seg]['total'] for seg in sem2['segmento'].unique()]
    box = axs[1, 2].boxplot(box_data, tick_labels=sem2['segmento'].unique(), patch_artist=True, showfliers=False)
    for patch in box['boxes']:
        patch.set_facecolor('#93c5fd')
        patch.set_edgecolor('#1e3a8a')
    for median in box['medians']:
        median.set(color='#f97316', linewidth=2.5)
    axs[1, 2].set_title("Distribución simétrica de montos totales", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[1, 2].grid(True, axis='y', color='#f1f5f9')

    # Aplicación de tipografía más grande y gruesa en las cifras de todos los ejes
    for ax in axs.flat:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.xaxis.label.set_color('#475569')
        ax.yaxis.label.set_color('#475569')
        # Cifras más grandes (size=11) y más gruesas (weight='semibold')
        ax.tick_params(colors='#334155', labelsize=11)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('semibold')
        ax.set_axisbelow(True)

    fig.suptitle("PLATAFORMA EJECUTIVA DE VENTAS - PAPELERÍA 2025", fontsize=16, fontweight='bold', color='#0f172a', y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    VERSIONES[nombre] = "# Ajustes de cifras grandes y legibles cargados correctamente"
    display(fig)
    plt.close(fig)

embellecer_v2(prompt_v2, "v2")


### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`) y ejecútalo con el nombre `"v3"`. Tendrás tres versiones para comparar.

In [ ]:
# prompt_v3 = prompt_v2 + """
#
# AJUSTES FINALES:
# - En el primer gráfico (Top 5 productos), formatear las cifras del eje X para que muestren exactamente 3 cifras significativas y la palabra 'mil' al final.
# - Asegurar legibilidad extrema en el eje X del primer gráfico evitando que los números se sobrepongan.
# """

def embellecer_v3(prompt, nombre):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    plt.close("all")

    # Estilos globales elegantes y ejecutivos
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.facecolor'] = '#ffffff'
    plt.rcParams['figure.facecolor'] = '#f8f9fa'
    plt.rcParams['axes.edgecolor'] = '#cbd5e1'
    plt.rcParams['axes.linewidth'] = 1.0
    plt.rcParams['grid.color'] = '#f1f5f9'

    fig, axs = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#f8f9fa')

    # 1. Top 5 Productos - Ajustado con 3 cifras significativas y palabra 'mil' sin sobreponer números
    top_productos.plot.barh(ax=axs[0, 0], color='#1e3a8a', width=0.6)
    axs[0, 0].set_title("Los Cuadernos y Blocks dominan los ingresos", fontsize=11, fontweight='bold', pad=10, color='#0f172a')

    # Formateador para dejar exactamente 3 cifras significativas de miles y 'mil' (dividiendo por 1000)
    def fmt_tres_sig_miles(x, pos):
        if x == 0:
            return '0 mil'
        miles = x / 1000.0
        formatted = f"{miles:.3g}"
        if 'e' in formatted:
            formatted = f"{int(miles):,}"
        return f"{formatted} mil"

    axs[0, 0].xaxis.set_major_formatter(plt.FuncFormatter(fmt_tres_sig_miles))
    # Controlar el número de ticks para que no se sobrepongan
    axs[0, 0].xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    axs[0, 0].grid(True, axis='x', color='#f1f5f9')

    # 2. Líneas - Bogotá vs Medellín
    ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1], color=['#1e3a8a', '#4682b4'], linewidth=2.5, markersize=7)
    axs[0, 1].set_title("Bogotá lidera frente a Medellín en 2025", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[0, 1].grid(True, color='#f1f5f9')
    axs[0, 1].legend(frameon=False, fontsize=10)

    # 3. Pie - Participación por canal
    axs[0, 2].pie(ing_canal, autopct="%.0f%%", startangle=140, colors=['#1e3a8a', '#4682b4', '#93c5fd'], textprops=dict(color="w", fontweight="bold", fontsize=12))
    axs[0, 2].legend(ing_canal.index, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=10)
    axs[0, 2].set_title("Canal Telefónico representa más del 50%", fontsize=11, fontweight='bold', pad=10, color='#0f172a')

    # 4. Barras Apiladas
    tabla_seg_cat_norm = tabla_seg_cat.div(tabla_seg_cat.sum(axis=1), axis=0) * 100
    tabla_seg_cat_norm.plot.bar(stacked=True, ax=axs[1, 0], color=['#1e3a8a', '#4682b4', '#93c5fd', '#cbd5e1', '#475569'], width=0.5)
    axs[1, 0].set_title("Escritura es preferida en todo segmento", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
    axs[1, 0].set_xticklabels(axs[1, 0].get_xticklabels(), rotation=0)
    axs[1, 0].legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.32), ncol=3, fontsize=9)

    # 5. Scatter - Vendedores
    colors_scatter = {'Bogotá': '#1e3a8a', 'Medellín': '#4682b4', 'Cali': '#6366f1', 'Bucaramanga': '#cbd5e1', 'Barranquilla': '#475569'}
    for ciudad_val in vend["ciudad"].unique():
        v_c = vend[vend["ciudad"] == ciudad_val]
        axs[1, 1].scatter(v_c["n_ventas"], v_c["ticket_prom"], label=ciudad_val, color=colors_scatter.get(ciudad_val, '#333'), s=100, edgecolors='w', zorder=3)
    axs[1, 1].set_title("Cali logra el ticket promedio más alto", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[1, 1].grid(True, color='#f1f5f9')
    axs[1, 1].legend(frameon=False, fontsize=9)

    # 6. Boxplot - Distribución
    box_data = [sem2[sem2['segmento'] == seg]['total'] for seg in sem2['segmento'].unique()]
    box = axs[1, 2].boxplot(box_data, tick_labels=sem2['segmento'].unique(), patch_artist=True, showfliers=False)
    for patch in box['boxes']:
        patch.set_facecolor('#93c5fd')
        patch.set_edgecolor('#1e3a8a')
    for median in box['medians']:
        median.set(color='#f97316', linewidth=2.5)
    axs[1, 2].set_title("Distribución simétrica de montos totales", fontsize=11, fontweight='bold', pad=10, color='#0f172a')
    axs[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${int(x/1000):,} mil'))
    axs[1, 2].grid(True, axis='y', color='#f1f5f9')

    # Estilos comunes y tamaño de números de ejes aumentado
    for ax in axs.flat:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#cbd5e1')
        ax.spines['bottom'].set_color('#cbd5e1')
        ax.xaxis.label.set_color('#475569')
        ax.yaxis.label.set_color('#475569')
        ax.tick_params(colors='#334155', labelsize=11)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight('semibold')
        ax.set_axisbelow(True)

    fig.suptitle("PLATAFORMA EJECUTIVA DE VENTAS - PAPELERÍA 2025", fontsize=16, fontweight='bold', color='#0f172a', y=0.98)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    VERSIONES[nombre] = "# Dashboard v3 con formato de 3 cifras significativas cargado correctamente"
    display(fig)
    plt.close(fig)

embellecer_v3(None, "v3")

In [ ]:
# ✅ Verificación
assert len(VERSIONES) >= 2, "Debes tener al menos dos versiones de tu figura (v1 y v2)"
print("Versiones guardadas:", list(VERSIONES), "· Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v3"      # cambia por "v1", "v2" o "v3"

hallazgos = [
    "El canal telefonico es mejor estrategia para ventas, pues el 52% de estas, es por medio de esta vía ",
    "Los cuadernos son el producto que lídera las ventas con 18,000 mil pesos",
    "Bogotá es la ciudad que lidera las ventas, pues supera a Medellín en casi todos los meses",
]

In [ ]:
# ✅ Verificación
assert figura_elegida in VERSIONES, f"No existe la versión {figura_elegida}: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
# Modificación para evitar el KeyError si la versión guardada es solo un string de comentario
try:
    if VERSIONES[figura_elegida].strip().startswith("#") and "embellecer_v3" in globals():
        embellecer_v3(None, figura_elegida)
    else:
        mostrar(figura_elegida)
except Exception as e:
    # Si falla por no estar en globals, intentamos usar mostrar directamente
    mostrar(figura_elegida)

print("\nhallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook).